In [7]:
# !pip install tensorflow
import tensorflow as tf
from tensorflow.keras import layers, models

def build_cnn_autoencoder(input_shape=(1024, 64, 3)):
    """
    정상 3상 전류 STFT 이미지를 학습하여 고장을 탐지하기 위한 CNN 오토인코더
    """
    # ==========================================
    # 1. 인코더 (Encoder) : 이미지의 핵심 특징만 압축 추출
    # ==========================================
    inputs = layers.Input(shape=input_shape)
    
    # 특징을 찾고(Conv2D) -> 크기를 절반으로 줄임(MaxPooling2D)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2), padding='same')(x) # (512, 32, 32)
    
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x) # (256, 16, 16)
    
    x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(x)
    encoded = layers.MaxPooling2D((2, 2), padding='same')(x) # (128, 8, 8) 
    # ↑ 여기가 가장 압축된 병목(Bottleneck) 지점입니다.

    # ==========================================
    # 2. 디코더 (Decoder) : 압축된 특징을 바탕으로 원본 이미지 복원
    # ==========================================
    x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(encoded)
    x = layers.UpSampling2D((2, 2))(x) # 크기를 다시 2배로 키움 (256, 16, 8)
    
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x) # (512, 32, 16)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x) # (1024, 64, 32)
    
    # 마지막 복원층: 채널을 원본과 똑같이 3채널(R,G,B)로 맞추고, 
    # 데이터가 0~1 사이 정규화 값이므로 sigmoid 활성화 함수 사용
    decoded = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

    # ==========================================
    # 3. 전체 모델 조립 및 컴파일
    # ==========================================
    autoencoder = models.Model(inputs, decoded)
    
    # 원본 이미지와 복원된 이미지 간의 픽셀 오차(Mean Squared Error)를 최소화하도록 학습
    autoencoder.compile(optimizer='adam', loss='mse')
    
    return autoencoder

# 모델 생성 및 구조 확인
autoencoder = build_cnn_autoencoder()
autoencoder.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 1024, 64, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 1024, 64, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 512, 32, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 512, 32, 16)    │         4,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 256, 16, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 256, 16, 8)     │         1,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 128, 8, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 128, 8, 8)      │           584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_6 (UpSampling2D)  │ (None, 256, 16, 8)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 256, 16, 16)    │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_7 (UpSampling2D)  │ (None, 512, 32, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 512, 32, 32)    │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_8 (UpSampling2D)  │ (None, 1024, 64, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 1024, 64, 3)    │           867 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,939 (54.45 KB)

 Trainable params: 13,939 (54.45 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
import numpy as np
import glob

# 1. 학습 데이터(.npy) 로드
input_files = glob.glob('normal_image/*.npy')
X_train_list = []
# print(np.load(input_files[0]))

for input_file in input_files:
    img = np.load(input_file)

    img_cropped = img[:1024, :, :]
    X_train_list.append(img_cropped)
X_train = np.array(X_train_list)
# print(X_train)
print(f"데이터 로드 완료 / shape: {X_train.shape}")

# 2. 모델 학습
history = autoencoder.fit(
    x=X_train, 
    y=X_train,           # 정답은 입력값 자기 자신
    epochs=20,           # 학습 반복 횟수
    batch_size=8,        # 한 번에 투입할 이미지 장 수
    validation_split=0.2 # 검증용 모의고사 데이터 비율 (20%)
)
print("학습 완료")


# 3. 학습된 정상 데이터들의 복원 오차 확인
reconstructed_images = autoencoder.predict(X_train)
mse = tf.keras.losses.mse(X_train, reconstructed_images)
reconstruction_errors = np.mean(mse, axis=(1,2)) 
max_normal_error = np.max(reconstruction_errors)
print(f"정상 데이터들의 최대 복원 오차율: {max_normal_error:.6f}")

데이터 로드 완료 / shape: (320, 1024, 64, 3)
Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - loss: 9.1160e-04 - val_loss: 0.0010
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 94ms/step - loss: 9.1188e-04 - val_loss: 0.0010
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 9.0336e-04 - val_loss: 9.1798e-04
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - loss: 8.9487e-04 - val_loss: 9.1875e-04
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - loss: 9.0184e-04 - val_loss: 9.1955e-04
Epoch 6/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - loss: 9.0784e-04 - val_loss: 9.1150e-04
Epoch 7/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - loss: 8.7589e-04 - val_loss: 8.9285e-04
Epoch 8/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - loss: 8.9521e-04 - val_loss: 8.9550e-04
Epoch 9/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 110ms/step - loss: 8.6150e-04 - val_loss: 8.9732e-04
Epoch 10/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - loss: 8.7337e-04 - val_loss: 9.9046e-04
Epoch 11/20
32/32 ━━━━━━━━━━━━━━━━━